# 可读科研 Notebook 模板：计算、溯源与独立重绘

这是人工示例数据的教学模板，不代表论文算法或真实科研结论。

安装仓库根目录 `requirements-example.txt` 中的依赖。将本文件与 `example_measurements.csv` 放在同一目录，以该目录为工作目录后 Run All。默认图内使用英文以便跨平台运行，代码旁注使用中文；中文标签需选用实际安装的 CJK 字体。

仅重绘：重新启动 Kernel 后，只运行“4. 独立绘图”代码单元。它读取保存的结果和 manifest，不运行计算。更改科学输入或带 `science` 标签的代码后，旧结果会被拒绝。重命名 Notebook 时同步修改两个入口的 `NOTEBOOK_FILE`。

## 1. 科学输入与可追溯配置

科学参数集中在此。分组标准差采用 ddof=1；单样本组的 SD 无定义，保留 NaN。

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd

DATA_FILE = Path("example_measurements.csv")
NOTEBOOK_FILE = Path("notebook_template.ipynb")
OUTPUT_DIR = Path("example_output")
PARAMETERS = {"group_column": "group", "value_column": "value", "ddof": 1,
              "missing_policy": "exclude-empty-group-or-nonfinite-value"}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = OUTPUT_DIR / "group_summary.csv"

# 仅对科学单元取指纹，修改绘图样式不使计算缓存失效。
def science_hash(path):
    notebook = json.loads(path.read_text(encoding="utf-8"))
    source = ["".join(cell["source"]) for cell in notebook["cells"]
              if "science" in cell.get("metadata", {}).get("tags", [])]
    return hashlib.sha256(json.dumps(source, ensure_ascii=False).encode()).hexdigest()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


## 2. 输入检查与 QC

标识符按字符串读取；不把无效值变成零。保存被排除记录及原因，数据不足时停止。

In [ ]:
raw = pd.read_csv(DATA_FILE, dtype=str, keep_default_na=False)
required = {"sample_id", PARAMETERS["group_column"], PARAMETERS["value_column"]}
if not required <= set(raw.columns):
    raise ValueError(f"Missing columns: {required - set(raw.columns)}")
if raw["sample_id"].str.strip().eq("").any() or raw["sample_id"].duplicated().any():
    raise ValueError("Sample IDs must be nonempty and unique")
values = pd.to_numeric(raw[PARAMETERS["value_column"]], errors="coerce")
missing_group = raw[PARAMETERS["group_column"]].str.strip().eq("")
finite = np.isfinite(values)
excluded = raw.loc[missing_group | ~finite].copy()
excluded["qc_reason"] = [
    ";".join(reason for condition, reason in ((missing_group.loc[i], "empty_group"),
                                              (not finite.loc[i], "invalid_or_nonfinite_value"))
             if condition) for i in excluded.index]
excluded.to_csv(OUTPUT_DIR / "excluded_records.csv", index=False)
valid = raw.loc[~missing_group & finite].copy()
valid[PARAMETERS["value_column"]] = values.loc[valid.index]
if valid.empty:
    raise ValueError("No valid observations")
print(f"Input={len(raw)}, valid={len(valid)}, excluded={len(excluded)}")
display(valid)


## 3. 核心计算、保存与来源记录

示例仅演示分组均值、样本 SD，不代表任何领域算法验证。保存输入身份、参数、代码身份、环境及结果哈希。

In [ ]:
if PARAMETERS["ddof"] != 1:
    raise ValueError("This teaching implementation only supports sample SD with ddof=1")
summary = valid.groupby(PARAMETERS["group_column"], sort=True)[PARAMETERS["value_column"]].agg(
    n="count", mean="mean", sd="std"
).reset_index().rename(columns={PARAMETERS["group_column"]: "group"})
assert int(summary["n"].sum()) == len(valid)
assert np.isfinite(summary["mean"]).all()
assert summary.loc[summary["n"] == 1, "sd"].isna().all()
summary.to_csv(SUMMARY_CSV, index=False)
# 与示例 CSV 一起使用时的明确期望；不将此断言套用到真实数据。
if sha256(DATA_FILE) == "fb3b70cb0b2d724d518eb02673bd30714da8d34ded4be366a5762ec50518a887":
    np.testing.assert_allclose(summary["mean"], [3.0, 5.5, 8.0])
    np.testing.assert_allclose(summary["sd"], [1.0, 1.0, 1.0])
manifest = {
    "schema_version": "1.0", "synthetic_example": True,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input": {"path": str(DATA_FILE.resolve()), "sha256": sha256(DATA_FILE)},
    "scientific_parameters": PARAMETERS, "science_code_sha256": science_hash(NOTEBOOK_FILE),
    "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
    "result": {"path": SUMMARY_CSV.name, "sha256": sha256(SUMMARY_CSV),
               "rows": len(summary), "observations": len(valid)},
    "qc": {"input_rows": len(raw), "excluded_rows": len(excluded)}
}
(OUTPUT_DIR / "provenance.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
display(summary)
print("Example checks passed; scientific method validation still requires a real baseline.")


## 4. 独立绘图：干净 Kernel 只运行下一单元

输入路径、当前科学参数和样式在本单元明确可见；参数快照须与计算阶段一致。若仅有归档结果且原始输入不可得，会保留历史来源并明确未重新验证输入。SD 无定义时不画误差条，不修改统计表中的 NaN。

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ===== 绘图输入身份；更改科学参数需回到计算阶段显式重算 =====
OUTPUT_DIR = Path("example_output")
NOTEBOOK_FILE = Path("notebook_template.ipynb")
EXPECTED_PARAMETERS = {"group_column": "group", "value_column": "value", "ddof": 1,
                       "missing_policy": "exclude-empty-group-or-nonfinite-value"}
# ===== 只影响图形的参数 =====
FIGSIZE = (7.0, 4.5)
FONT_SIZE = 11
DPI = 300
FONT_FAMILY = "DejaVu Sans"  # 中文标签可改为已安装的 Noto Sans CJK SC / PingFang SC
TITLE = "Synthetic group mean ± sample SD"
Y_LABEL = "Example measurement (arbitrary units)"
COLOR = "#4878A8"
FORMATS = ("svg", "pdf", "png")

manifest_path = OUTPUT_DIR / "provenance.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
result_path = OUTPUT_DIR / manifest["result"]["path"]
assert hashlib.sha256(result_path.read_bytes()).hexdigest() == manifest["result"]["sha256"], "Result changed; stop and investigate"
assert EXPECTED_PARAMETERS == manifest["scientific_parameters"], "Science parameters changed; explicitly recompute"
notebook = json.loads(NOTEBOOK_FILE.read_text(encoding="utf-8"))
science_sources = ["".join(cell["source"]) for cell in notebook["cells"]
                   if "science" in cell.get("metadata", {}).get("tags", [])]
current_hash = hashlib.sha256(json.dumps(science_sources, ensure_ascii=False).encode()).hexdigest()
assert current_hash == manifest["science_code_sha256"], "Science code changed; explicitly recompute"
input_path = Path(manifest["input"]["path"])
input_verified = input_path.is_file()
if input_verified:
    assert hashlib.sha256(input_path.read_bytes()).hexdigest() == manifest["input"]["sha256"], "Input changed; explicitly recompute"
else:
    print("Archived result: original input unavailable; input identity NOT revalidated.")
plot_data = pd.read_csv(result_path, dtype={"group": str}, keep_default_na=False)
for column in ("n", "mean", "sd"):
    plot_data[column] = pd.to_numeric(plot_data[column], errors="raise") if column != "sd" else pd.to_numeric(plot_data[column], errors="coerce")
assert len(plot_data) > 0 and len(plot_data) == manifest["result"]["rows"], "Unexpected plot row count"
assert plot_data["group"].is_unique and plot_data["group"].str.strip().ne("").all()
assert np.isfinite(plot_data["mean"]).all(), "No valid means to plot"
assert (plot_data["n"] >= 1).all() and (plot_data["n"] % 1 == 0).all()
assert int(plot_data["n"].sum()) == manifest["result"]["observations"]
assert plot_data.loc[plot_data["n"] == 1, "sd"].isna().all()
assert np.isfinite(plot_data.loc[plot_data["n"] > 1, "sd"]).all()
assert (plot_data.loc[plot_data["n"] > 1, "sd"] >= 0).all()
font_manager.findfont(font_manager.FontProperties(family=FONT_FAMILY), fallback_to_default=False)
mpl.rcParams.update({"font.family": FONT_FAMILY, "svg.fonttype": "none", "pdf.fonttype": 42})

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    fig, ax = plt.subplots(figsize=FIGSIZE)
    x = np.arange(len(plot_data))
    ax.bar(x, plot_data["mean"], color=COLOR, edgecolor="black", linewidth=0.7)
    has_sd = np.isfinite(plot_data["sd"])
    if has_sd.any():
        ax.errorbar(x[has_sd], plot_data.loc[has_sd, "mean"], yerr=plot_data.loc[has_sd, "sd"],
                    fmt="none", color="black", capsize=4)
    labels = [f"{row.group}\n(n={int(row.n)})" + ("\nSD undefined" if row.n == 1 else "")
              for row in plot_data.itertuples()]
    ax.set_xticks(x, labels, fontsize=FONT_SIZE)
    ax.set_xlabel("Example group", fontsize=FONT_SIZE)
    ax.set_ylabel(Y_LABEL, fontsize=FONT_SIZE)
    ax.set_title(TITLE, fontsize=FONT_SIZE + 1)
    fig.tight_layout()
    fig.canvas.draw()
    glyph = [str(item.message) for item in caught if "Glyph" in str(item.message) and "missing" in str(item.message)]
    if glyph:
        raise RuntimeError(f"Missing glyphs: {glyph[:3]}; configure an installed font")
    for extension in FORMATS:
        fig.savefig(OUTPUT_DIR / f"group_mean_example.{extension}", dpi=DPI, bbox_inches="tight")
    glyph = [str(item.message) for item in caught if "Glyph" in str(item.message) and "missing" in str(item.message)]
    assert not glyph, f"Missing glyphs while saving: {glyph[:3]}"
# 其他警告不静默丢弃，供研究者判断。
for item in caught:
    print(f"Render warning: {item.message}")
plot_data.to_csv(OUTPUT_DIR / "figure_data.csv", index=False)
style = {"figsize": FIGSIZE, "font_size": FONT_SIZE, "dpi": DPI, "font_family": FONT_FAMILY,
         "title": TITLE, "y_label": Y_LABEL, "color": COLOR, "formats": FORMATS}
(OUTPUT_DIR / "figure_parameters.json").write_text(json.dumps(style, indent=2), encoding="utf-8")
figure_provenance = {"result_sha256": manifest["result"]["sha256"],
                     "science_code_sha256": current_hash, "input_revalidated": input_verified,
                     "matplotlib": mpl.__version__,
                     "plot_code_sha256": hashlib.sha256(json.dumps(["".join(c["source"]) for c in notebook["cells"] if "plot" in c.get("metadata", {}).get("tags", [])]).encode()).hexdigest(),
                     "outputs": {ext: hashlib.sha256((OUTPUT_DIR / f"group_mean_example.{ext}").read_bytes()).hexdigest() for ext in FORMATS}}
(OUTPUT_DIR / "figure_provenance.json").write_text(json.dumps(figure_provenance, indent=2), encoding="utf-8")
fields = "schema_version,dataset_id,source_code_version,refactor_code_version,method,check_type,sample_key,metric,old_value,new_value,abs_diff,rel_diff,atol,rtol,comparison_rule,qc_before,qc_after,status,notes".split(",")
with (OUTPUT_DIR / "figure_validation.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    for metric in ("nonempty", "valid-values", "no-glyph-warning", "data-rows", "visual-check"):
        row = dict.fromkeys(fields, "")
        row.update(schema_version="1.0", dataset_id="synthetic-example", method="group-summary",
                   check_type="figure", metric=f"figure:{metric}",
                   status="NOT-RUN" if metric == "visual-check" else "PASS",
                   comparison_rule="Review labels and clipping manually" if metric == "visual-check" else "Assertions in independent plot cell",
                   notes="group_mean_example.png; figure_provenance.json")
        writer.writerow(row)
plt.show()


## 5. 交付边界

`example_output/` 包含统计表、QC 排除表、计算与图件来源、绘图数据、样式参数、SVG/PDF/PNG 以及统一格式的图件检查表。

示例的数值与自动绘图检查通过，不代表论文方法复现通过。`figure:visual-check` 默认 NOT-RUN，实际打开图件检查标签与裁切后才可填写 PASS。真实项目还需原实现对照、单位与样本设计核验、方法来源和容差依据。